In [1]:
#### Statistical Tests and Analyses
from scipy.stats import shapiro, ttest_rel, levene, wilcoxon, mannwhitneyu, ranksums
import pandas as pd
from pathlib import Path
import sys

project_root = Path().resolve().parent.parent  # from inside src/analyze/

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

INSTA_FILE = project_root / "data/transformed/insta_final.csv"
POP_FILE = project_root / "data/transformed/pop_final.csv"
TT_FILE = project_root / "data/transformed/tt_final_country.csv"
FB_FILE = project_root / "data/transformed/fb_final.csv" 

FILE_NAMES = {INSTA_FILE: "Instagram", POP_FILE: "Population", TT_FILE: "TikTok", FB_FILE: "Facebook"}
FILES = [INSTA_FILE, TT_FILE, FB_FILE, POP_FILE]


In [2]:
#### Statistical Tests
#perform Shapiro-Wilk test for normality, H0=normal distribution, If the p-value of the test > α = .05, then the data is assumed to be normally distributed
for file in FILES:
    name = FILE_NAMES[file]
    df = pd.read_csv(file,sep=";")
    data = df["total_ratio"]
    stat, p = shapiro(data)
    print(f"Shapiro's test for {name}:{stat:.4f},{p:.4f}")
    if p > 0.05:
        print(f"{name}: Data is likely normally distributed (p > 0.05)\n")
    else:
        print(f"{name}: Data is not normally distributed (p ≤ 0.05)\n")



Shapiro's test for Instagram:0.9507,0.0000
Instagram: Data is not normally distributed (p ≤ 0.05)

Shapiro's test for TikTok:0.9803,0.0195
TikTok: Data is not normally distributed (p ≤ 0.05)

Shapiro's test for Facebook:0.9187,0.0000
Facebook: Data is not normally distributed (p ≤ 0.05)

Shapiro's test for Population:0.7702,0.0000
Population: Data is not normally distributed (p ≤ 0.05)



In [3]:
#test homogeneity of variance, H0=equal variance, homoscedastic, if we get a p-value > 0.05 we can assume that our data is heteroscedastic (which is what we want)
#p>0.05, we would fail to reject the null hypothesis. We do not have sufficient evidence to say that the variance in the ratio between the data sampls is significantly different, i.e. equal variance.
fb = pd.read_csv(FB_FILE,sep=";")
tt = pd.read_csv(TT_FILE,sep=";")
pop = pd.read_csv(POP_FILE,sep=";")
inst = pd.read_csv(INSTA_FILE,sep=";")

fb = fb["total_ratio"].dropna()
tt = tt["total_ratio"].dropna()
pop = pop["total_ratio"].dropna()
inst = inst["total_ratio"].dropna()

stat, p = levene(fb,tt,pop,inst)

print(f"Levene's test:{stat:.4f}, {p:.4f}")
if p > 0.05:
    print("Variances are likely equal (homogeneous).")
else:
    print("Variances are not equal (not homogeneous).")


Levene's test:38.0780, 0.0000
Variances are not equal (not homogeneous).


In [5]:
# Perform Wilcoxon test if assumptions are not met --> need to have the same length
fb = pd.read_csv(FB_FILE,sep=";")
tt = pd.read_csv(TT_FILE,sep=";")
pop = pd.read_csv(POP_FILE,sep=";")
inst = pd.read_csv(INSTA_FILE,sep=";")

fb = fb[["iso3","total_ratio","total_ratio_std"]].dropna()
tt = tt[["iso3","total_ratio","total_ratio_std"]].dropna()
pop = pop[["iso3","total_ratio"]].dropna()
inst = inst[["iso3","total_ratio","total_ratio_std"]].dropna()

common_iso3 = (
    set(fb["iso3"]) &
    set(tt["iso3"]) &
    set(pop["iso3"]) &
    set(inst["iso3"])
)

# make all dataframes equal length
fb_trim = fb[fb["iso3"].isin(common_iso3)].copy()
tt_trim = tt[tt["iso3"].isin(common_iso3)].copy()
pop_trim = pop[pop["iso3"].isin(common_iso3)].copy()
inst_trim = inst[inst["iso3"].isin(common_iso3)].copy()

fb_ttl = fb_trim["total_ratio"]
tt_ttl = tt_trim["total_ratio"]
pop_ttl = pop_trim["total_ratio"]
inst_ttl = inst_trim["total_ratio"]

t_statistic_tf, p_value_tf = wilcoxon(tt_ttl, fb_ttl)
t_statistic_tp, p_value_tp = wilcoxon(tt_ttl, pop_ttl)
t_statistic_ti, p_value_ti = wilcoxon(tt_ttl, inst_ttl)

# Output the results
print(f"======Results on Country Total Gender Gap Ratio ==== \n")
print(f"Wilcoxon-statistic for Tiktok vs Facebook: {t_statistic_tf:.4f}. P-value: {p_value_tf:.4f}")
if p_value_tf < 0.05:
    print(f"The difference between TikTok and Facebook is statistically significant (p < 0.05).\n")
else:
    print(f"No statistically significant difference between TikTok and Facebook (p > 0.05).\n")

print(f"Wilcoxon-statistic for Tiktok vs Population: {t_statistic_tp:.4f}. P-value: {p_value_tp:.4f}")
if p_value_tp < 0.05:
    print(f"The difference between TikTok and Population is statistically significant (p < 0.05).\n")
else:
    print(f"No statistically significant difference between TikTok and Population (p > 0.05).\n")

print(f"Wilcoxon-statistic for Tiktok vs Instagram: {t_statistic_ti:.4f}. P-value: {p_value_ti:.4f}")
if p_value_ti < 0.05:
    print(f"The difference between TikTok and Instagram is statistically significant (p < 0.05).\n")
else:
    print(f"No statistically significant difference between TikTok and Instagram (p > 0.05).\n")

fb_std = fb_trim["total_ratio_std"]
tt_std = tt_trim["total_ratio_std"]
inst_std = inst_trim["total_ratio_std"]

t_statistic_tf, p_value_tf = wilcoxon(tt_std, fb_std)
t_statistic_tp, p_value_tp = wilcoxon(tt_std, pop_ttl)
t_statistic_ti, p_value_ti = wilcoxon(tt_std, inst_std)

# Output the results
print(f"======Results on Country Total Gender Gap Standardized Ratio ==== \n")
print(f"Wilcoxon-statistic for Tiktok vs Facebook: {t_statistic_tf:.4f}. P-value: {p_value_tf:.4f}")
if p_value_tf < 0.05:
    print(f"The difference between TikTok and Facebook is statistically significant (p < 0.05).\n")
else:
    print(f"No statistically significant difference between TikTok and Facebook (p > 0.05).\n")

print(f"Wilcoxon-statistic for Tiktok vs Population: {t_statistic_tp:.4f}. P-value: {p_value_tp:.4f}")
if p_value_tp < 0.05:
    print(f"The difference between TikTok and Population is statistically significant (p < 0.05).\n")
else:
    print(f"No statistically significant difference between TikTok and Population (p > 0.05).\n")

print(f"Wilcoxon-statistic for Tiktok vs Instagram: {t_statistic_ti:.4f}. P-value: {p_value_ti:.4f}")
if p_value_ti < 0.05:
    print(f"The difference between TikTok and Instagram is statistically significant (p < 0.05).\n")
else:
    print(f"No statistically significant difference between TikTok and Instagram (p > 0.05).\n")


======Results on Country Total Gender Gap Ratio ==== 

Wilcoxon-statistic for Tiktok vs Facebook: 4926.5000. P-value: 0.0791
No statistically significant difference between TikTok and Facebook (p > 0.05).

Wilcoxon-statistic for Tiktok vs Population: 4750.5000. P-value: 0.0079
The difference between TikTok and Population is statistically significant (p < 0.05).

Wilcoxon-statistic for Tiktok vs Instagram: 3195.0000. P-value: 0.0000
The difference between TikTok and Instagram is statistically significant (p < 0.05).

======Results on Country Total Gender Gap Standardized Ratio ==== 

Wilcoxon-statistic for Tiktok vs Facebook: 4839.0000. P-value: 0.0555
No statistically significant difference between TikTok and Facebook (p > 0.05).

Wilcoxon-statistic for Tiktok vs Population: 4132.0000. P-value: 0.0002
The difference between TikTok and Population is statistically significant (p < 0.05).

Wilcoxon-statistic for Tiktok vs Instagram: 3208.0000. P-value: 0.0000
The difference between TikTok